# Il futuro nei numeri: serie temporali e forecasting

Il codice del capitolo [«Il futuro nei numeri: serie temporali e forecasting»](https://book.paithon.it/main/SerieTemporali/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision

## Il futuro nei numeri: serie temporali e forecasting

[Leggi la pagina](https://book.paithon.it/main/SerieTemporali/overview.html)


### Perché è un problema diverso (e difficile)


In [ ]:
import numpy as nprng = np.random.default_rng(0)n = 200t = np.arange(n)# serie sintetica: tendenza + stagionalità (periodo 12) + rumoreserie = 0.05 * t + 2.0 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.5, n)def autocorr(x, lag):    x = x - x.mean()    return np.sum(x[lag:] * x[:-lag]) / np.sum(x * x)print(f"autocorrelazione a lag 1:  {autocorr(serie, 1):.3f}")print(f"autocorrelazione a lag 12: {autocorr(serie, 12):.3f}")# rimescolando l'ordine, la dipendenza temporale svaniscemescolata = rng.permutation(serie)print(f"lag 1 dopo lo shuffle:     {autocorr(mescolata, 1):.3f}")

## Componenti e modelli classici: da ARIMA a Holt-Winters

[Leggi la pagina](https://book.paithon.it/main/SerieTemporali/componenti-e-classici.html)


### In pratica: stimare un AR(1) ai minimi quadrati


In [ ]:
import numpy as nprng = np.random.default_rng(42)# --- genera una serie dal modello AR(1): x_t = c + phi * x_{t-1} + rumore ---phi_vero, c_vero, sigma = 0.6, 4.0, 1.0n = 500x = np.zeros(n)x[0] = c_vero / (1 - phi_vero)                 # parte dalla media di lungo periodo (10)for t in range(1, n):    x[t] = c_vero + phi_vero * x[t - 1] + rng.normal(0, sigma)# --- stima ai minimi quadrati: regredisci x_t su [1, x_{t-1}] ---y = x[1:]                                       # bersaglio: x_tXmat = np.column_stack([np.ones(n - 1), x[:-1]])  # colonne: costante e x_{t-1}beta, *_ = np.linalg.lstsq(Xmat, y, rcond=None)   # risolve i minimi quadratic_hat, phi_hat = betaprint(f"phi vero = {phi_vero:.2f}   phi stimato = {phi_hat:.3f}")print(f"c vero   = {c_vero:.2f}   c stimato   = {c_hat:.3f}")# --- previsione one-step dopo l'ultima osservazione ---x_next = c_hat + phi_hat * x[-1]print(f"ultima osservazione x_T = {x[-1]:.3f}")print(f"previsione   x_(T+1)    = {x_next:.3f}")

## Validare e rappresentare: backtesting e feature temporali

[Leggi la pagina](https://book.paithon.it/main/SerieTemporali/validazione-e-feature.html)


### In pratica: walk-forward e MASE con NumPy


In [ ]:
import numpy as npdef walk_forward_split(n, min_train, horizon):    """Split cronologico a finestra espansa (walk-forward / backtesting):    restituisce coppie (indici_train, indici_test) col test sempre nel futuro."""    for t in range(min_train, n - horizon + 1, horizon):        yield np.arange(t), np.arange(t, t + horizon)def mase(y_vero, y_pred, y_train, m=1):    """MASE: MAE del modello sul test, scalato sul MAE del naive a passo m    calcolato in-sample sul training."""    errore_modello = np.mean(np.abs(y_vero - y_pred))    errore_naive_train = np.mean(np.abs(y_train[m:] - y_train[:-m]))    return errore_modello / errore_naive_train# --- serie sintetica: trend leggero + stagionalità settimanale + rumore ---rng = np.random.default_rng(0)n, m = 140, 7t = np.arange(n)serie = 10 + 0.05 * t + 3 * np.sin(2 * np.pi * t / m) + rng.normal(0, 0.4, n)mase_stagionale, mase_semplice = [], []for idx_train, idx_test in walk_forward_split(n, min_train=28, horizon=m):    storia, futuro = serie[idx_train], serie[idx_test]    pred_stagionale = storia[-m:]            # naive stagionale: ripeti l'ultima settimana    pred_semplice = np.full(m, storia[-1])   # naive semplice: ripeti l'ultimo valore    # denominatore MASE sempre col naive a passo 1 sul training    mase_stagionale.append(mase(futuro, pred_stagionale, storia, m=1))    mase_semplice.append(mase(futuro, pred_semplice, storia, m=1))print(f"iterazioni di walk-forward: {len(mase_stagionale)}")print(f"MASE medio - naive stagionale: {np.mean(mase_stagionale):.3f}")print(f"MASE medio - naive semplice:   {np.mean(mase_semplice):.3f}")

## Forecasting neurale: da RNN ai Transformer e ai foundation model

[Leggi la pagina](https://book.paithon.it/main/SerieTemporali/forecasting-neurale.html)


### DeepAR: una rete per mille serie, e una distribuzione


In [ ]:
import numpy as nprng = np.random.default_rng(0)# A ogni passo il "modello" predice media e deviazione del prossimo valore.def prossimo(x_prec):    mu = 4.0 + 0.6 * x_prec     # parte deterministica (media condizionata)    sigma = 1.0                 # incertezza a un passo    return mu, sigma# Previsione probabilistica a 5 passi per CAMPIONAMENTO ANCESTRALE:# molte traiettorie, ciascuna reinietta il proprio campione come input.orizzonte, n_traj = 5, 20000x_T = 12.0traj = np.zeros((n_traj, orizzonte))for j in range(n_traj):    x = x_T    for h in range(orizzonte):        mu, sigma = prossimo(x)        x = rng.normal(mu, sigma)   # si CAMPIONA, non si prende la media        traj[j, h] = x# Dai campioni ricaviamo i quantili: la banda di previsione.q10, q50, q90 = np.percentile(traj, [10, 50, 90], axis=0)for h in range(orizzonte):    print(f"t+{h+1}:  mediana {q50[h]:5.2f}   banda 80% [{q10[h]:5.2f}, {q90[h]:5.2f}]")

### In pratica: una TCN in PyTorch


In [ ]:
import torchimport torch.nn as nnclass Taglia(nn.Module):    """Rimuove gli ultimi `n` istanti: preserva la causalità."""    def __init__(self, n):        super().__init__()        self.n = n    def forward(self, x):                 # x: (batch, canali, tempo)        return x[:, :, :-self.n].contiguous() if self.n > 0 else xclass BloccoTCN(nn.Module):    def __init__(self, c_in, c_out, kernel=3, dilation=1):        super().__init__()        pad = (kernel - 1) * dilation      # padding causale (a sinistra nel tempo)        self.conv = nn.Conv1d(c_in, c_out, kernel, padding=pad, dilation=dilation)        self.taglia = Taglia(pad)          # elimina il padding di troppo a destra        self.relu = nn.ReLU()        # connessione residua: adatta i canali con una conv 1x1 se necessario        self.giu = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else None    def forward(self, x):        y = self.relu(self.taglia(self.conv(x)))   # uscita causale, stessa lunghezza        r = x if self.giu is None else self.giu(x)        return self.relu(y + r)class TCN(nn.Module):    def __init__(self, c_in=1, canali=32, kernel=3, n_blocchi=3):        super().__init__()        strati = []        for i in range(n_blocchi):            d = 2 ** i                     # dilatazione 1, 2, 4, ...            ci = c_in if i == 0 else canali            strati.append(BloccoTCN(ci, canali, kernel, dilation=d))        self.rete = nn.Sequential(*strati)        self.testa = nn.Linear(canali, 1)  # dall'ultimo istante -> previsione    def forward(self, x):                  # x: (batch, tempo), serie univariata        h = self.rete(x.unsqueeze(1))      # (batch, canali, tempo)        return self.testa(h[:, :, -1])     # ultimo istante -> (batch, 1)